# 🤖 SQL Agent — How It Works

**Blauw-Zwart Analytics Pipeline — Presentation Walkthrough**

This notebook walks through the complete SQL agent pipeline step by step:
an English-language question enters, a validated PostgreSQL query is generated
and executed, and a Markdown answer comes back — all without the user ever
writing SQL.

---

### Pipeline at a glance

```
User question
      │
      ▼
┌─────────────────────────────────────────────────────┐
│                  Flask API  (app.py)                │
│  POST /api/ask/stream  →  AgentRequest              │
└──────────────────────┬──────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────┐
│               run_ask()  (graph.py)                 │
│                                                     │
│  ┌──────────────────────────────────────────────┐  │
│  │          Primary Agent  (ReAct loop)         │  │
│  │  get_semantic_layer → list_tables            │  │
│  │  → describe_table → search_columns           │  │
│  │  → sample_table   → execute_select ✓         │  │
│  └──────────────┬───────────────────────────────┘  │
│                 │ fails?                            │
│                 ▼                                   │
│  ┌──────────────────────────────────────────────┐  │
│  │    Repair Agent  (one-shot, stronger model)  │  │
│  │    describe_table → execute_select ✓         │  │
│  └──────────────────────────────────────────────┘  │
└──────────────────────┬──────────────────────────────┘
                       │
          ┌────────────┴────────────┐
          ▼                         ▼
   SQL Guardrails            PostgreSQL
   strip_fences()            llm_reader role
   rewrite_schema()          LIMIT 50
   validate_sql()            statement_timeout 10 s
          │
          ▼
   Markdown answer  →  SSE stream  →  Browser
```

**Key libraries:**
| Library | Role |
|---|---|
| `langchain` / `langgraph` | ReAct agent loop and tool-calling framework |
| `langchain-openrouter` | Hosted LLM access (DeepSeek, Grok, …) |
| `sqlglot` | SQL parsing, AST validation, and schema rewriting |
| `psycopg2` | PostgreSQL execution as read-only `llm_reader` role |
| `flask` | HTTP/SSE API layer |

---
## 0 · Setup

Load environment variables (API key, database URL) and make sure the
`frontend_app` source package is on the path.

**Postgres from this notebook (on your machine):** Compose already publishes Postgres to
`localhost:${POSTGRES_PORT}`. URLs that use hostname `postgres` only work *inside* Docker.
Set `JUPYTER_SQL_AGENT_USE_LOCALHOST=1` in `.env` (see `.env.example`) so the next cell
rewrites `postgres:5432` → `localhost:<POSTGRES_PORT>` for this kernel. Use `0` if your
kernel runs inside Compose and should keep `postgres`.

In [1]:
import os
import sys
from urllib.parse import quote, urlsplit, urlunsplit

from dotenv import load_dotenv


def _postgres_service_url_to_localhost(url: str, published_port: str) -> str:
    """Rewrite @postgres:5432 → @localhost:<published_port> for host-side notebooks."""
    parts = urlsplit(url)
    if (parts.hostname or "").lower() != "postgres":
        return url
    if (parts.port or 5432) != 5432:
        return url
    user, pw = parts.username or "", parts.password
    if user:
        auth = quote(user, safe="")
        if pw is not None:
            auth += ":" + quote(pw, safe="")
        netloc = f"{auth}@localhost:{published_port}"
    else:
        netloc = f"localhost:{published_port}"
    return urlunsplit((parts.scheme, netloc, parts.path, parts.query, parts.fragment))


# Load .env from repo root (one level above notebooks/)
load_dotenv(dotenv_path="../.env")

_flag = os.getenv("JUPYTER_SQL_AGENT_USE_LOCALHOST", "").strip().lower()
if _flag in ("1", "true", "yes"):
    _port = (os.getenv("POSTGRES_PORT") or "5432").strip() or "5432"
    for _key in ("LLM_READER_DATABASE_URL", "DATABASE_URL"):
        _v = os.getenv(_key)
        if _v:
            os.environ[_key] = _postgres_service_url_to_localhost(_v, _port)

# Make frontend_app importable when running the notebook directly
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("OPENROUTER_API_KEY set:", bool(os.getenv("OPENROUTER_API_KEY")))
_db_url_set = bool(
    os.getenv("LLM_READER_DATABASE_URL") or os.getenv("DATABASE_URL")
)
print("DATABASE_URL set:      ", _db_url_set)
if _flag in ("1", "true", "yes"):
    _pg_port = (os.getenv("POSTGRES_PORT") or "5432").strip() or "5432"
    print(
        "JUPYTER_SQL_AGENT_USE_LOCALHOST: rewrote postgres -> localhost "
        f"for DB URLs (port {_pg_port})"
    )

OPENROUTER_API_KEY set: True
DATABASE_URL set:       True


---
## 1 · LLM Connection via OpenRouter

The agent uses `ChatOpenRouter` — a LangChain-compatible chat model that routes
requests to any model available on [openrouter.ai](https://openrouter.ai).
The model ID is configurable at runtime via the settings page (stored in
`llm_config.json`) without restarting the server.

In [2]:
from langchain_core.messages import HumanMessage
from langchain_openrouter import ChatOpenRouter

# This is the same factory the agent uses internally (providers.build_chat_model)
MODEL_ID = os.getenv("OPENROUTER_AGENT_MODEL", "mistralai/mistral-small-2603")

llm = ChatOpenRouter(
    model=MODEL_ID,
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# Quick sanity-check: plain text generation (no tools yet)
response = llm.invoke([HumanMessage(content="Reply with exactly: SQL agent ready.")])
print(f"Model: {MODEL_ID}")
print(f"Response: {response.content}")

Model: mistralai/mistral-small-2603
Response: SQL agent ready.


---
## 2 · Schema Discovery Tools

The agent has **no hard-coded schema knowledge**. Instead it discovers the
warehouse on demand by calling read-only tools. This keeps the system prompt
small and handles schema drift automatically.

| Tool | Purpose |
|---|---|
| `get_semantic_layer()` | Subjects, metrics, join paths, answer-style rules |
| `list_tables()` | All relations in the `dbt_dev` schema with their dbt layer |
| `describe_table(table)` | Columns, types, and descriptions for one table |
| `search_columns(pattern)` | Find columns by name across all tables |
| `sample_table(table, limit)` | Peek at a few rows to verify shape |
| `execute_select(sql)` | The **only** way to run SQL — goes through guardrails first |

In [3]:
import json

from frontend_app.sql_agent.tools import describe_table, list_tables, search_columns

# list_tables: discover each dbt_dev relation as JSON rows,
# or return an {"error": ...} payload on failure
raw = list_tables.invoke({})
tables_json = getattr(raw, "content", raw)  # LangChain may return str or a message object
tables = json.loads(tables_json)

if isinstance(tables, dict) and tables.get("error"):
    raise RuntimeError(tables["error"])

if not isinstance(tables, list):
    raise TypeError(
        "Expected a JSON list from list_tables, got "
        f"{type(tables).__name__}: {tables!r}"
    )

print(f"Found {len(tables)} tables/views in dbt_dev:")
for t in tables:
    if not isinstance(t, dict):
        print(f"  (skip non-dict row: {t!r})")
        continue
    if t.get("_truncated"):
        print(f"  ... {t.get('_message', 'truncated')}")
        continue
    print(f"  [{t['layer']:12s}]  {t['name']}")

RuntimeError: list_tables failed: could not translate host name "postgres" to address: Name or service not known


In [ ]:
# describe_table: inspect the mart the agent uses most for fan analytics
desc_json = describe_table.invoke({"table": "mart_fan_loyalty"})
desc = json.loads(desc_json)

print(f"Table: {desc['name']}  ({desc['layer']})")
print(f"Description: {desc['description']}")
print(f"\nColumns ({len(desc['columns'])})")
print(f"{'Name':<30} {'Type':<20} Description")
print("-" * 80)
for col in desc["columns"]:
    print(f"{col['name']:<30} {col['data_type']:<20} {col['description'][:40]}")

In [ ]:
# search_columns: find all columns related to "spend" across every table
results_json = search_columns.invoke({"pattern": "spend", "limit": 10})
results = json.loads(results_json)

print("Columns matching '%spend%':")
for r in results:
    print(f"  {r['table']}.{r['column']} ({r['data_type']})  — {r['description'][:50]}")

---
## 3 · SQL Guardrails

Before any SQL reaches the database it passes through **three layers** of
sanitisation:

1. **`_strip_fences`** — removes markdown code fences that some models wrap SQL in
2. **`_rewrite_layer_schema_qualifiers`** — maps dbt layer names (`marts.x`,
   `staging.x`) to the actual Postgres schema `dbt_dev.x`
3. **`_validate_sql`** — two-pass validation:
   - *sqlglot AST check*: parse with the Postgres dialect, reject DDL/DML nodes
     (`INSERT`, `UPDATE`, `DELETE`, `DROP`, …), multi-statement input
   - *Regex belt-and-braces*: catch any mutating keyword the AST might miss

In [ ]:
from frontend_app.sql_agent.guardrails import (
    _rewrite_layer_schema_qualifiers,
    _strip_fences,
    _validate_sql,
)

# --- strip_fences: models sometimes wrap SQL in markdown code blocks ---
raw_from_llm = """
```sql
SELECT fan_id, total_spend FROM marts.mart_fan_loyalty ORDER BY total_spend DESC LIMIT 5;
```
"""

step1 = _strip_fences(raw_from_llm)
print("After _strip_fences:")
print(f"  {step1!r}")

# --- rewrite_layer_schema_qualifiers: marts.X → dbt_dev.X ---
step2 = _rewrite_layer_schema_qualifiers(step1)
print("\nAfter _rewrite_layer_schema_qualifiers:")
print(f"  {step2!r}")

In [ ]:
# --- _validate_sql: this query is safe and passes all checks ---
safe_sql = "SELECT fan_id, total_spend FROM dbt_dev.mart_fan_loyalty ORDER BY total_spend DESC"

_validate_sql(safe_sql)
print("✅ Valid SQL — passed all guardrails")
print(f"   {safe_sql}")

In [ ]:
# --- _validate_sql: a mutating statement is caught and blocked ---
dangerous_examples = [
    "DROP TABLE mart_fan_loyalty",
    "SELECT * FROM fans; DELETE FROM fans",
    "INSERT INTO fans VALUES (1, 'hacker')",
]

for sql in dangerous_examples:
    try:
        _validate_sql(sql)
        print(f"⚠️  PASSED (unexpected): {sql[:60]}")
    except ValueError as exc:
        print(f"🛡️  Blocked: {sql[:50]!r}")
        print(f"   Reason: {exc}")
        print()

---
## 4 · Safe SQL Execution

Queries run as the **`llm_reader`** Postgres role — a read-only account with:
- `REVOKE INSERT, UPDATE, DELETE, TRUNCATE, ...` on all schemas
- `statement_timeout = '10s'` to abort runaway queries
- An outer `LIMIT 50` wrapper to cap result size

Even if the guardrails were somehow bypassed, the role itself cannot mutate data.

In [ ]:
from frontend_app.sql_agent.database import _execute_sql

sql = """
SELECT fan_id, total_spend, matches_attended
FROM dbt_dev.mart_fan_loyalty
ORDER BY total_spend DESC
"""

# _execute_sql wraps the query in: SELECT * FROM (<sql>) AS llm_query LIMIT 50
rows = _execute_sql(sql)

print(f"Returned {len(rows)} rows (hard-capped at 50)")
print("\nTop 5 fans by total spend:")
print(f"{'fan_id':<12} {'total_spend':>14} {'matches_attended':>18}")
print("-" * 46)
for row in rows[:5]:
    print(f"{row['fan_id']:<12} {row['total_spend']:>14.2f} {row['matches_attended']:>18}")

---
## 5 · The ReAct Agent Loop

The agent is built with LangGraph's `create_agent` — a **ReAct** (Reason +
Act) loop that alternates between:
1. **Reasoning** — the LLM decides which tool to call next
2. **Acting** — the tool runs and its result is appended to the message history

The loop continues until the LLM produces a final answer with no tool calls,
or the iteration cap is hit (`AGENT_MAX_TOOL_ITERATIONS`, default 8).

`AgentObservabilityHandler` (the callback registered in the `config`) logs
every LLM call and tool invocation with millisecond timing and the request
correlation ID.

In [ ]:
import logging

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

from frontend_app.sql_agent.observability import AgentObservabilityHandler
from frontend_app.sql_agent.prompts import AGENT_SYSTEM_PROMPT, build_user_prompt
from frontend_app.sql_agent.tools import ALL_TOOLS

# Enable INFO-level logs so we can watch the agent reason
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(name)s — %(message)s", force=True)

question = "Who are the top 3 fans by total spend, and how many matches did each attend?"

user_prompt = build_user_prompt(
    question=question,
    conversation_section="",
    answer_style_rules=None,
)

agent = create_agent(
    model=llm,
    tools=ALL_TOOLS,
    system_prompt=AGENT_SYSTEM_PROMPT,
)

print(f"Question: {question}")
print("\n" + "=" * 60)
print("Running agent (watch the INFO logs for tool calls)…")
print("=" * 60 + "\n")

handler = AgentObservabilityHandler()

state = agent.invoke(
    {"messages": [HumanMessage(content=user_prompt)]},
    config={"recursion_limit": 21, "callbacks": [handler]},
)

In [ ]:
# Inspect the full message trace produced by the agent
messages = state["messages"]

print(f"Total messages in conversation: {len(messages)}\n")
for i, msg in enumerate(messages):
    if isinstance(msg, HumanMessage):
        print(f"[{i}] 👤 User:  {str(msg.content)[:80]}…")
    elif isinstance(msg, AIMessage):
        if getattr(msg, "tool_calls", None):
            calls = ", ".join(tc["name"] for tc in msg.tool_calls)
            print(f"[{i}] 🤖 Agent → tool calls: {calls}")
        else:
            preview = str(msg.content)[:100].replace("\n", " ")
            print(f"[{i}] 🤖 Agent → final answer: {preview}…")
    elif isinstance(msg, ToolMessage):
        result_preview = str(msg.content)[:80].replace("\n", " ")
        print(f"[{i}]  🔧 Tool ({msg.name}): {result_preview}…")

---
## 6 · Full Pipeline: `run_ask()`

`run_ask()` is the single public entrypoint that orchestrates everything above:

- Builds the user prompt (with optional conversation context)
- Runs the primary agent
- Classifies the outcome (`success`, `no_sql`, `validation`, `execution`,
  `iteration_cap`)
- Optionally runs a **repair pass** on a stronger model if the primary failed
- Returns a typed `AgentResult` (success) or `AgentFailure` (unrecoverable)

In [ ]:
from frontend_app.sql_agent.graph import AgentRequest, AgentResult, run_ask

request = AgentRequest(
    question="What is the average total spend per fan who attended at least one match?",
)

print(f"Question: {request.question}\n")
outcome = run_ask(request)

print("\n" + "=" * 60)
if isinstance(outcome, AgentResult):
    print(f"✅  Success  (repaired={outcome.repaired})")
    print(f"   Model:    {outcome.agent_model}")
    print(f"   Rows:     {len(outcome.rows)}")
    print(f"\n   SQL used:\n   {outcome.sql}")
    print(f"\n   Answer:\n{outcome.answer}")
else:
    print(f"❌  Failure  phase={outcome.phase}")
    print(f"   {outcome.error}")

---
## 7 · The Repair Pass

When the primary agent fails (bad SQL, wrong table, iteration cap), a
**one-shot repair pass** runs automatically on the `repair_model` (which can
be set to a stronger, slower model in the settings).

The repair agent receives a condensed prompt with:
- The original question
- The SQL that failed
- The failure phase and error message

It has a smaller toolset (`describe_table` + `execute_select`) to stay focused.

Below we deliberately force a primary failure by injecting a broken SQL tool
response, then watch the repair pass take over.

In [ ]:
from frontend_app.sql_agent.llm_runtime_config import resolve_repair_model
from frontend_app.sql_agent.prompts import build_repair_user_prompt
from frontend_app.sql_agent.providers import build_chat_model
from frontend_app.sql_agent.tools import REPAIR_TOOLS

# Simulate the scenario where the primary agent produced invalid SQL
failed_sql   = "SELECT fan_id FROM staging.fans WHERE total_spendings > 100"
failure_phase   = "execution"
failure_message = 'relation \"staging.fans\" does not exist'
original_question = "List fans who spent more than €100"

print("Primary agent failed:")
print(f"  SQL:     {failed_sql}")
print(f"  Phase:   {failure_phase}")
print(f"  Error:   {failure_message}")

repair_prompt = build_repair_user_prompt(
    question=original_question,
    failed_sql=failed_sql,
    failure_phase=failure_phase,
    failure_message=failure_message,
    conversation_section="",
)

print("\n" + "=" * 60)
print("Repair prompt sent to the repair model:")
print("=" * 60)
print(repair_prompt)

In [ ]:
# Run the actual repair agent against the live database
from langchain.agents import create_agent

from frontend_app.sql_agent.prompts import REPAIR_SYSTEM_PROMPT

repair_model_id = resolve_repair_model(None)
repair_chat = build_chat_model(repair_model_id)

print(f"Repair model: {repair_model_id}")
print("Running repair pass…\n")

repair_agent = create_agent(
    model=repair_chat,
    tools=REPAIR_TOOLS,
    system_prompt=REPAIR_SYSTEM_PROMPT,
)

repair_state = repair_agent.invoke(
    {"messages": [HumanMessage(content=repair_prompt)]},
    config={"recursion_limit": 11, "callbacks": [AgentObservabilityHandler()]},
)

repair_messages = repair_state["messages"]
# Find the final AI text
final = next(
    (m.content for m in reversed(repair_messages)
     if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None)),
    "(no answer produced)",
)

print("\n" + "=" * 60)
print("✅  Repair pass answer:")
print("=" * 60)
print(final)

---
## Summary

| Stage | What happens |
|---|---|
| **1. LLM connection** | `ChatOpenRouter` connects to any hosted model |
| **2. Schema discovery** | Agent explores tables/columns on demand — no hard-coded schema |
| **3. Guardrails** | Fences stripped → schema rewritten → sqlglot AST + regex validation |
| **4. Safe execution** | `llm_reader` role + `LIMIT 50` + `statement_timeout 10s` |
| **5. ReAct loop** | Alternates tool calls and reasoning until SQL succeeds |
| **6. `run_ask()`** | Single entrypoint: returns typed `AgentResult` or `AgentFailure` |
| **7. Repair pass** | Stronger model retries with a focused toolset on failure |

The full stack is observable via structured logs with per-request correlation
IDs — set `LOG_LEVEL=DEBUG` in `.env` to see every LLM round-trip and DB
query with millisecond timing.